In [2]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-google-genai \
    google-genai \
    langsmith \
    pageindex \
    rank-bm25 \
    feedparser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth

In [27]:
import numpy as np
import os
from google.colab import userdata
import time

from rank_bm25 import BM25Okapi

from pageindex import PageIndexClient
import pageindex.utils as utils

import nltk
import string
import getpass
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
import feedparser
import langchain
import langchain_classic
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import Field

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
pi_client = PageIndexClient(api_key=userdata.get('{PAGEINDEX'))

In [5]:
doc=pi_client.submit_document("/content/20_news_articles (1).pdf")

In [6]:
doc_id=doc['doc_id']

In [7]:
while not pi_client.is_retrieval_ready(doc_id):
    print("Still indexing...")
    time.sleep(5)


Still indexing...
Still indexing...
Still indexing...


In [8]:
tree = pi_client.get_tree(doc_id)
print("Document Tree:")
utils.print_tree(tree)

Document Tree:
{'doc_id': 'pi-cmq1eep5u00k801qxvbqxt2eb',
 'status': 'completed',
 'retrieval_ready': True,
 'result': [{'title': 'The Dawn of Ambient Intelligence: How Ed...', 'node_id': '0000'},
            {'title': 'Sub-Saharan Manufacturing Boom Signals S...', 'node_id': '0001'},
            {'title': 'Deep Space Telescope Detects Atmospheric...', 'node_id': '0002'},
            {'title': 'Synthetic Enzyme Trials Offer Breakthrou...', 'node_id': '0003'},
            {'title': 'Spongy Cities: How Biophilic Design Pres...', 'node_id': '0004'},
            {'title': 'Deep Geothermal Energy Reaches Commercia...', 'node_id': '0005'},
            {'title': 'Sovereign Wealth Funds Pivot to Regenera...', 'node_id': '0006'},
            {'title': 'Acoustic Restoration Revives Degraded Co...', 'node_id': '0007'},
            {'title': 'Non-Invasive Neural Stimulation Shows Pr...', 'node_id': '0008'},
            {'title': 'The Rise of Symbolic AI: Bridging the Ga...', 'node_id': '0009'},
  

In [9]:
print(type(tree))

<class 'dict'>


In [10]:
tree

{'doc_id': 'pi-cmq1eep5u00k801qxvbqxt2eb',
 'status': 'completed',
 'retrieval_ready': True,
 'result': [{'title': 'The Dawn of Ambient Intelligence: How Edge Computing is Reshaping Daily Life',
   'node_id': '0000',
   'page_index': 1,
   'text': "# The Dawn of Ambient Intelligence: How Edge Computing is Reshaping Daily Life\n\nBy Marcus Vance • Published on June 1, 2026\n\nF decades, the promise of the smart home and the automated workplace relied heavily on centralized cloud servers.\n\nEvery voice command, every sensor trigger, and every automated routine had to travel thousands of miles to a data center and back. This architecture, while revolutionary for its time, introduced latency, privacy concerns, and a vulnerability to network outages. Today, a quiet revolution known as ambient intelligence, powered by decentralized edge computing, is fundamentally transforming our relationship with technology.\n\nAmbient intelligence refers to electronic environments that are sensitive and 

In [11]:
for root in tree:
  print(root)

doc_id
status
retrieval_ready
result
metadata


In [12]:
i=0
for nodes in tree["result"]:
  print(nodes)

{'title': 'The Dawn of Ambient Intelligence: How Edge Computing is Reshaping Daily Life', 'node_id': '0000', 'page_index': 1, 'text': "# The Dawn of Ambient Intelligence: How Edge Computing is Reshaping Daily Life\n\nBy Marcus Vance • Published on June 1, 2026\n\nF decades, the promise of the smart home and the automated workplace relied heavily on centralized cloud servers.\n\nEvery voice command, every sensor trigger, and every automated routine had to travel thousands of miles to a data center and back. This architecture, while revolutionary for its time, introduced latency, privacy concerns, and a vulnerability to network outages. Today, a quiet revolution known as ambient intelligence, powered by decentralized edge computing, is fundamentally transforming our relationship with technology.\n\nAmbient intelligence refers to electronic environments that are sensitive and responsive to the presence of people. Unlike traditional systems that require explicit user input—such as tapping 

In [13]:
documents=[]

for nodes in tree["result"]:
    documents.append(
        {
          "node_id":nodes["node_id"],
          "title":nodes["title"],

        }
    )

In [14]:
tokenized_doc = [doc["title"].split() for doc in documents]

In [15]:
bm25 = BM25Okapi(tokenized_doc)

In [16]:
langchain_docs = []
for doc in documents:
    langchain_docs.append(Document(
                            page_content=doc["title"],
                            metadata={
                                      "title": doc["title"],
                                      "node_id": doc["node_id"]
                                    }))


In [17]:
class BM25Retriever(BaseRetriever):
    bm25: BM25Okapi = Field(...)
    documents: list = Field(...)

    def _get_relevant_documents(self, query: str):
        query_tokens = query.lower().split()
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:3]
        return [self.documents[i] for i in top_indices]


retriever = BM25Retriever(
    bm25=bm25,
    documents=langchain_docs
)

In [18]:
documents

[{'node_id': '0000',
  'title': 'The Dawn of Ambient Intelligence: How Edge Computing is Reshaping Daily Life'},
 {'node_id': '0001',
  'title': 'Sub-Saharan Manufacturing Boom Signals Shift in Global Supply Chains'},
 {'node_id': '0002',
  'title': 'Deep Space Telescope Detects Atmospheric Anomalies on Proxima B'},
 {'node_id': '0003',
  'title': 'Synthetic Enzyme Trials Offer Breakthrough in Microplastic Degredation'},
 {'node_id': '0004',
  'title': 'Spongy Cities: How Biophilic Design Preserved Cities During the Spring Floods'},
 {'node_id': '0005',
  'title': 'Deep Geothermal Energy Reaches Commercial Viability in Deep Drill Trials'},
 {'node_id': '0006',
  'title': 'Sovereign Wealth Funds Pivot to Regenerative Agriculture Infrastructure'},
 {'node_id': '0007',
  'title': 'Acoustic Restoration Revives Degraded Coral Reefs in the South Pacific'},
 {'node_id': '0008',
  'title': 'Non-Invasive Neural Stimulation Shows Promise in Reversing Age-Related Memory Loss'},
 {'node_id': '0009

In [20]:
results = retriever.invoke(
    ""
)

for doc in results:
    print(doc.metadata['title'])

Quantum Sensor Array Detects Elusive Cosmic Neutrino Background
Mycelium Packaging Replaces Polystyrene in Major Electronics Shipments
Digital Detox Reserves: The Emerging Economy of absolute Off-Grid Tourism


In [23]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [24]:
prompt = ChatPromptTemplate.from_template("""You are a question-answering assistant.
Answer ONLY using the provided context.
If the answer cannot be found in the context, say:
\"I could not find the answer in the provided documents.\"
Context:
{context}
Question:
{input}
Answer:"""
)

In [33]:
document_chain= create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)
response = retrieval_chain.invoke({
    "input": "Which countries were highlighted as experiencing rapid manufacturing growth due to the AfCFTA framework?"
})

In [34]:
print(response['answer'])

I could not find the answer in the provided documents.


In [35]:
full_corpus = "\n\n".join(
    [doc.page_content for doc in langchain_docs]
)

In [46]:
start = time.time()

response = llm.invoke(
    f"""
Context:
{full_corpus}

Question:
Which countries were highlighted as experiencing rapid manufacturing growth due to the AfCFTA framework??
"""
)

naive_time = time.time() - start

print(response.content)
print("Time:", naive_time)

The provided text mentions a "Sub-Saharan Manufacturing Boom Signals Shift in Global Supply Chains," but it **does not** specify which countries are experiencing this growth or explicitly link it to the AfCFTA framework.
Time: 1.7923166751861572


In [47]:
start = time.time()

response = retrieval_chain.invoke({"input": "Which countries were highlighted as experiencing rapid manufacturing growth due to the AfCFTA framework?"})
rag_time = time.time()-start

print(response["answer"])
print("Time:", rag_time)

I could not find the answer in the provided documents.
Time: 1.5420894622802734
